## Классификация фигур
### Студент: Шайдуров Даниил Сергеевич

In [77]:
from pathlib import Path
import os

import open3d as o3d
import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [78]:
# Проверить доступность CUDA в Open3D
print("CUDA available:", o3d.core.cuda.is_available())

# Использовать CUDA-тензоры если доступно
if o3d.core.cuda.is_available():
    device = o3d.core.Device("CUDA:0")
    # Конвертировать point cloud в CUDA тензор

CUDA available: True


In [79]:
sphere_path = 'dataset/test/sphere/s_0003.ply'
cyl_path =  'dataset/test/cylinder/y_0003.ply' 
cube_path = 'dataset/test/cube/c_0003.ply' 
cone_path = 'dataset/test/cone/o_0003.ply' 
torus_path = 'dataset/test/torus/t_0003.ply' 

In [80]:
pcd_raw  = o3d.io.read_point_cloud(torus_path)
o3d.visualization.draw_geometries([pcd_raw])

In [81]:
# С шумом 
sphere_path = 'dataset_noisy/test/sphere/s_0003.ply'
cyl_path =  'dataset_noisy/test/cylinder/y_0003.ply' 
cube_path = 'dataset_noisy/test/cube/c_0003.ply' 
cone_path = 'dataset_noisy/test/cone/o_0003.ply' 
torus_path = 'dataset_noisy/test/torus/t_0003.ply' 

In [82]:
pcd_raw  = o3d.io.read_point_cloud(torus_path)
o3d.visualization.draw_geometries([pcd_raw])

In [83]:
DATA_ROOT = Path("dataset_noisy")
SPLITS = ["train", "val", "test"]
CLASSES = ["sphere", "cube", "cylinder", "cone", "torus"]
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Параметры препроцессинга для признаков

VOXEL_SIZE = 0.005 # 0.005
RADIUS_NORMAL = 4.0 * VOXEL_SIZE # попробовать 2
MAX_NN_NORMAL = 50

# FPFH
RADIUS_FPFH = 8.0 * VOXEL_SIZE
MAX_NN_FPFH = 150

# ISS keypoints
ISS_SALIENT_RADIUS = 6.0 * VOXEL_SIZE
ISS_NON_MAX_RADIUS = 4.0 * VOXEL_SIZE
ISS_GAMMA_21 = 0.975
ISS_GAMMA_32 = 0.975
ISS_MIN_NEIGHBORS = 5

In [84]:
def load_point_cloud(path: Path) -> o3d.geometry.PointCloud:
    pcd = o3d.io.read_point_cloud(str(path))
    return pcd

In [85]:
def quick_prepare(pcd: o3d.geometry.PointCloud,
                  voxel_size: float = None,
                  radius_normal: float = RADIUS_NORMAL,
                  max_nn_normal: int = MAX_NN_NORMAL) -> o3d.geometry.PointCloud:

    if voxel_size:
        pcd = pcd.voxel_down_sample(voxel_size)
        
    pcd.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=max_nn_normal)
    )
    pcd.orient_normals_consistent_tangent_plane(max_nn_normal)
    return pcd

### FPFH

In [86]:
def compute_fpfh_vector(pcd: o3d.geometry.PointCloud,
                        radius_feature: float = RADIUS_FPFH,
                        max_nn: int = MAX_NN_FPFH) -> np.ndarray:
    """Возвращает вектор признаков объекта: concat(mean, std) FPFH."""
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=max_nn)
    ).data  # shape: (33, n_points)

    mu = fpfh.mean(axis=1)
    sd = fpfh.std(axis=1)
    return np.concatenate([mu, sd], axis=0).astype(np.float32)  # 66-мерный вектор

In [87]:
def iter_split_files(root: Path, split: str):
    base = root / split
    for cls in CLASSES:
        cls_dir = base / cls
        if not cls_dir.exists():
            continue
        for fn in os.listdir(cls_dir):
            if fn.lower().endswith((".ply", ".pcd", ".xyz")):
                yield cls, (cls_dir / fn)

def extract_vector_for_object(path: Path) -> np.ndarray:
    pcd = load_point_cloud(path)
    pcd = quick_prepare(pcd, VOXEL_SIZE, RADIUS_NORMAL, MAX_NN_NORMAL)

    return compute_fpfh_vector(pcd, RADIUS_FPFH, MAX_NN_FPFH)

def build_split_matrix(root: Path, split: str):
    X, y, ids = [], [], []
    for cls, path in tqdm(
        list(iter_split_files(root, split)), desc=f"{split}", unit="obj"
        ):
        try:
            vec = extract_vector_for_object(path)
            X.append(vec)
            y.append(CLS_TO_IDX[cls])
            ids.append(str(path))
        except Exception as e:
            print(f"[WARN] {path}: {e}")
    if len(X) == 0:
        return np.empty((0,)), np.empty((0,)), []
    X = np.stack(X)
    y = np.array(y, dtype=np.int64)
    return X, y, ids

In [88]:
def train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, model, label: str):
    model.fit(Xtr, ytr)
    for split_name, Xs, ys in [("val", Xval, yval), ("test", Xte, yte)]:
        yp = model.predict(Xs)
        acc = accuracy_score(ys, yp)
        print(f"[{label}] {split_name} accuracy: {acc:.4f}")
        print(classification_report(ys, yp, target_names=CLASSES, digits=4))
        print("Confusion:\n", confusion_matrix(ys, yp))
        print("-"*60)

In [89]:
feature_sets = {}
for split in SPLITS:
    X, y, ids = build_split_matrix(DATA_ROOT, split)
    feature_sets[split] = {"X": X, "y": y, "ids": ids}
    print(split, X.shape, np.bincount(y, minlength=len(CLASSES)))

train: 100%|██████████| 250/250 [01:59<00:00,  2.10obj/s]


train (250, 66) [50 50 50 50 50]


val: 100%|██████████| 75/75 [00:35<00:00,  2.13obj/s]


val (75, 66) [15 15 15 15 15]


test: 100%|██████████| 75/75 [00:35<00:00,  2.11obj/s]

test (75, 66) [15 15 15 15 15]


In [90]:
Xtr = feature_sets["train"]["X"]
ytr = feature_sets["train"]["y"]
Xval = feature_sets["val"]["X"]
yval = feature_sets["val"]["y"]
Xte  = feature_sets["test"]["X"]
yte  = feature_sets["test"]["y"]

In [91]:
# KNN
knn = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    KNeighborsClassifier(n_neighbors=7, metric="euclidean", weights="distance")
)
train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, knn, f"KNN")

# SVM (RBF)
svm = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    SVC(kernel="rbf", C=5.0, gamma='auto', class_weight='balanced')
)
train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, svm, f"SVM-RBF")

[KNN] val accuracy: 0.3600
              precision    recall  f1-score   support

      sphere     0.8000    0.2667    0.4000        15
        cube     0.2791    0.8000    0.4138        15
    cylinder     0.3158    0.4000    0.3529        15
        cone     1.0000    0.1333    0.2353        15
       torus     0.5000    0.2000    0.2857        15

    accuracy                         0.3600        75
   macro avg     0.5790    0.3600    0.3375        75
weighted avg     0.5790    0.3600    0.3375        75

Confusion:
 [[ 4  5  4  0  2]
 [ 0 12  3  0  0]
 [ 0  9  6  0  0]
 [ 0  8  4  2  1]
 [ 1  9  2  0  3]]
------------------------------------------------------------
[KNN] test accuracy: 0.4933
              precision    recall  f1-score   support

      sphere     1.0000    0.3333    0.5000        15
        cube     0.4286    1.0000    0.6000        15
    cylinder     0.3636    0.5333    0.4324        15
        cone     1.0000    0.0667    0.1250        15
       torus     0.66

# ISS (Intrinsic Shape Signatures):

In [92]:
def compute_iss_keypoints(pcd: o3d.geometry.PointCloud,
                          salient_radius: float = ISS_SALIENT_RADIUS,
                          non_max_radius: float = ISS_NON_MAX_RADIUS,
                          gamma_21: float = ISS_GAMMA_21,
                          gamma_32: float = ISS_GAMMA_32,
                          min_neighbors: int = ISS_MIN_NEIGHBORS) -> o3d.geometry.PointCloud:
    kp = o3d.geometry.keypoint.compute_iss_keypoints(
        pcd,
        salient_radius=salient_radius,
        non_max_radius=non_max_radius,
        gamma_21=gamma_21,
        gamma_32=gamma_32,
        min_neighbors=min_neighbors
    )
    return kp

In [93]:
sphere = quick_prepare(load_point_cloud(sphere_path))
cube = quick_prepare(load_point_cloud(cube_path))
cyl = quick_prepare(load_point_cloud(cyl_path))
cone = quick_prepare(load_point_cloud(cone_path))
torus = quick_prepare(load_point_cloud(torus_path))

skp = compute_iss_keypoints(sphere)
ckp = compute_iss_keypoints(cube)
cykp = compute_iss_keypoints(cyl)
cokp = compute_iss_keypoints(cone)
tkp = compute_iss_keypoints(torus)

In [94]:
skp

PointCloud with 111 points.

In [95]:
o3d.visualization.draw_geometries([skp])
o3d.visualization.draw_geometries([ckp])
o3d.visualization.draw_geometries([cykp])
o3d.visualization.draw_geometries([cokp])
o3d.visualization.draw_geometries([tkp])

In [96]:
fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        skp,
        o3d.geometry.KDTreeSearchParamHybrid(radius=RADIUS_FPFH, max_nn=MAX_NN_FPFH)
    )

In [97]:
def iter_split_files(root: Path, split: str):
    base = root / split
    for cls in CLASSES:
        cls_dir = base / cls
        if not cls_dir.exists():
            continue
        for fn in os.listdir(cls_dir):
            if fn.lower().endswith((".ply", ".pcd", ".xyz")):
                yield cls, (cls_dir / fn)

def extract_vector_for_object(path: Path) -> np.ndarray:
    """descriptor: 'fpfh' | 'shot' | 'iss_fpfh'."""
    pcd = load_point_cloud(path)
    pcd = quick_prepare(pcd, VOXEL_SIZE, RADIUS_NORMAL, MAX_NN_NORMAL)
    kp = compute_iss_keypoints(pcd)
    return compute_fpfh_vector(kp, RADIUS_FPFH, MAX_NN_FPFH)
    # return compute_iss_fpfh_vector(pcd)

def build_split_matrix(root: Path, split: str):
    X, y, ids = [], [], []
    for cls, path in tqdm(list(iter_split_files(root, split)),
                          desc=f"{split}", unit="obj"):
        try:
            vec = extract_vector_for_object(path)
            X.append(vec)
            y.append(CLS_TO_IDX[cls])
            ids.append(str(path))
        except Exception as e:
            print(f"[WARN] {path}: {e}")
    if len(X) == 0:
        return np.empty((0,)), np.empty((0,)), []
    X = np.stack(X)
    y = np.array(y, dtype=np.int64)
    return X, y, ids

In [98]:
feature_sets = {}
for split in SPLITS:
    X, y, ids = build_split_matrix(DATA_ROOT, split)
    feature_sets[split] = {"X": X, "y": y, "ids": ids}
    print(split, X.shape, np.bincount(y, minlength=len(CLASSES)))
    
Xtr = feature_sets["train"]["X"]
ytr = feature_sets["train"]["y"]
Xval = feature_sets["val"]["X"]
yval = feature_sets["val"]["y"]
Xte  = feature_sets["test"]["X"]
yte  = feature_sets["test"]["y"]

train:   0%|          | 0/250 [00:00<?, ?obj/s]

train: 100%|██████████| 250/250 [01:52<00:00,  2.22obj/s]


train (250, 66) [50 50 50 50 50]


val: 100%|██████████| 75/75 [00:28<00:00,  2.60obj/s]


val (75, 66) [15 15 15 15 15]


test: 100%|██████████| 75/75 [00:28<00:00,  2.63obj/s]

test (75, 66) [15 15 15 15 15]


In [99]:
# KNN
knn = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    KNeighborsClassifier(n_neighbors=7, metric="euclidean", weights="distance")
)
train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, knn, f"KNN")

# SVM (RBF)
svm = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    SVC(kernel="rbf", C=10.0, gamma="scale")
)
train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, svm, f"SVM-RBF")

/home/daniil/Learn/Semestr_3/3D/HomeWork/HomeWork_4/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/daniil/Learn/Semestr_3/3D/HomeWork/HomeWork_4/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/daniil/Learn/Semestr_3/3D/HomeWork/HomeWork_4/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

[KNN] val accuracy: 0.3200
              precision    recall  f1-score   support

      sphere     0.3846    0.3333    0.3571        15
        cube     0.3824    0.8667    0.5306        15
    cylinder     0.1538    0.1333    0.1429        15
        cone     0.0000    0.0000    0.0000        15
       torus     0.2667    0.2667    0.2667        15

    accuracy                         0.3200        75
   macro avg     0.2375    0.3200    0.2595        75
weighted avg     0.2375    0.3200    0.2595        75

Confusion:
 [[ 5  1  4  0  5]
 [ 1 13  1  0  0]
 [ 1  8  2  0  4]
 [ 3  6  4  0  2]
 [ 3  6  2  0  4]]
------------------------------------------------------------
[KNN] test accuracy: 0.4133
              precision    recall  f1-score   support

      sphere     0.5385    0.4667    0.5000        15
        cube     0.4000    0.8000    0.5333        15
    cylinder     0.1667    0.2000    0.1818        15
        cone     0.0000    0.0000    0.0000        15
       torus     0.64

### Выводы:

SVM-RBF показал значительно лучшие результаты (70-72% accuracy), чем KNN (45-57%), и является более стабильной моделью. KNN демонстрирует неудовлетворительное качество классификации.